Memory Coalescing & Access Patterns

In [30]:
#Implèmenter  deux matrice  addition kernels 
from numba import cuda
import numpy as np
import time

@cuda.jit
def matrix_add_coalesced(a, b, out):
    x, y = cuda.grid(2)
    if y < a.shape[0] and x < a.shape[1]:
        out[y, x] = a[y, x] + b[y, x]

@cuda.jit
def matrix_add_uncoalesced(a, b, out):
    x, y = cuda.grid(2)
    if x < a.shape[0] and y < a.shape[1]:
        out[x, y] = a[x, y] + b[x, y]

print("Q1 : deux kernels CUDA définis avec succès.")
print("GPU CUDA disponible :", cuda.is_available())

Q1 : deux kernels CUDA définis avec succès.
GPU CUDA disponible : True


In [31]:
if not cuda.is_available():
    raise RuntimeError(
        "Aucun GPU NVIDIA CUDA utilisable. Vérifie le pilote NVIDIA "
        "et que le noyau VS Code utilise le bon environnement Python."
    )

N = 8192
A = np.ones((N, N), dtype=np.float32)
B = np.full((N, N), 2.0, dtype=np.float32)

# Transfert CPU (host) -> GPU (device)
d_a = cuda.to_device(A)
d_b = cuda.to_device(B)
d_out = cuda.device_array((N, N), dtype=np.float32)

print(f"Q2 : A et B = {N} x {N}, dtype = {A.dtype}")
print(f"Taille de A : {A.nbytes / (1024**2):.0f} Mio")
print(f"Taille de B : {B.nbytes / (1024**2):.0f} Mio")
print(f"Mémoire GPU minimale pour A, B et sortie : {3 * A.nbytes / (1024**2):.0f} Mio")
print("Transfert vers le GPU terminé.")

Q2 : A et B = 8192 x 8192, dtype = float32
Taille de A : 256 Mio
Taille de B : 256 Mio
Mémoire GPU minimale pour A, B et sortie : 768 Mio
Transfert vers le GPU terminé.


In [32]:
threadsperblock = (32, 8)
blockspergrid = (
    (N + threadsperblock[0] - 1) // threadsperblock[0],
    (N + threadsperblock[1] - 1) // threadsperblock[1]
)

print("Q3 : dimensions du bloc :", threadsperblock)
print("Threads par bloc :", threadsperblock[0] * threadsperblock[1])
print("Dimensions de la grille :", blockspergrid)
print("Nombre total de blocs :", blockspergrid[0] * blockspergrid[1])

Q3 : dimensions du bloc : (32, 8)
Threads par bloc : 256
Dimensions de la grille : (256, 1024)
Nombre total de blocs : 262144


In [33]:
def benchmark_kernel(kernel, iterations=10):
    # Dry run : compilation JIT et première exécution, NON chronométrées
    kernel[blockspergrid, threadsperblock](d_a, d_b, d_out)
    cuda.synchronize()

    times = []
    for _ in range(iterations):
        start = time.perf_counter()
        kernel[blockspergrid, threadsperblock](d_a, d_b, d_out)
        cuda.synchronize()  # attendre que le kernel se termine réellement
        end = time.perf_counter()
        times.append(end - start)

    return sum(times) / len(times)  # moyenne en secondes


N_ITER = 10
time_coalesced = benchmark_kernel(matrix_add_coalesced, N_ITER)
time_uncoalesced = benchmark_kernel(matrix_add_uncoalesced, N_ITER)

print(f"Q4 : temps moyen coalesced   = {time_coalesced * 1000:.4f} ms")
print(f"Q4 : temps moyen uncoalesced = {time_uncoalesced * 1000:.4f} ms")
print(f"Nombre d'itérations mesurées par kernel : {N_ITER}")

Q4 : temps moyen coalesced   = 3.4104 ms
Q4 : temps moyen uncoalesced = 6.9793 ms
Nombre d'itérations mesurées par kernel : 10


In [34]:
speedup = time_uncoalesced / time_coalesced

print(f"Q5 : speedup mesuré = {speedup:.3f}x")
if speedup > 1:
    print(f"Le kernel coalesced est environ {speedup:.2f} fois plus rapide.")
elif speedup < 1:
    print("Sur cette mesure, le kernel coalesced n'est pas plus rapide.")
else:
    print("Les deux kernels ont le même temps moyen mesuré.")

Q5 : speedup mesuré = 2.046x
Le kernel coalesced est environ 2.05 fois plus rapide.


In [35]:
bytes_transferred = 3 * (N ** 2) * 4
bandwidth_coalesced = bytes_transferred / (time_coalesced * 1e9)
bandwidth_uncoalesced = bytes_transferred / (time_uncoalesced * 1e9)

print(f"Q6 : volume de données théorique = {bytes_transferred:,} octets")
print(f"Bande passante coalesced   = {bandwidth_coalesced:.3f} GB/s")
print(f"Bande passante uncoalesced = {bandwidth_uncoalesced:.3f} GB/s")

Q6 : volume de données théorique = 805,306,368 octets
Bande passante coalesced   = 236.135 GB/s
Bande passante uncoalesced = 115.385 GB/s


Why does accessing consecutive rows instead of columns slash throughput on the same GPU hardware? NumPy stocke les matrices en row-major. Dans matrix_add_coalesced, les threads voisins accèdent à des éléments contigus (a[y, x]), ce qui permet le memory coalescing et améliore le débit mémoire. Dans matrix_add_uncoalesced, les accès (a[x, y]) sont espacés en mémoire, ce qui génère plus de transactions et réduit généralement les performances. accès contigus = coalescing = plus rapide ; accès dispersés = moins efficace.

Shared Memory Banks & Bank Conflicts

In [36]:
from numba import cuda
import numpy as np
import time

@cuda.jit
def shared_bank_stride_test(d_out, stride):
    s = cuda.shared.array(1024, dtype=np.float32)
    tid = cuda.threadIdx.x

    # Initialize shared memory
    s[tid] = tid * 1.0
    cuda.syncthreads()

    # Strided read
    val = s[(tid * stride) % 1024]
    for i in range(1000):
        val += s[(tid * stride) % 1024]

    if tid == 0:
        d_out[0] = val

print("Q1 : kernel défini (il s'exécutera sur GPU à la question 2).")

Q1 : kernel défini (il s'exécutera sur GPU à la question 2).


In [37]:
if not cuda.is_available():
    raise RuntimeError("CUDA indisponible : vérifiez votre GPU NVIDIA, le pilote et le kernel Python de VS Code.")

threads_per_block = 32
blocks_per_grid = 1

d_out = cuda.device_array(1, dtype=np.float32)

# Un premier lancement avec stride = 1
shared_bank_stride_test[blocks_per_grid, threads_per_block](d_out, 1)
cuda.synchronize()

print("Q2 : blocs =", blocks_per_grid)
print("Q2 : threads/bloc =", threads_per_block)
print("Q2 : résultat du thread 0 =", d_out.copy_to_host()[0])

Q2 : blocs = 1
Q2 : threads/bloc = 32
Q2 : résultat du thread 0 = 0.0


c:\Users\GiGi\Desktop\TPAML\.venv\Lib\site-packages\numba_cuda\numba\cuda\dispatcher.py:748: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [38]:
strides = [1, 2, 4, 8, 16, 32]
iterations = 20
execution_times_ms = {}

for stride in strides:
    # Dry run : compilation / échauffement (hors mesures)
    shared_bank_stride_test[blocks_per_grid, threads_per_block](d_out, stride)
    cuda.synchronize()

    durations_ms = []
    for _ in range(iterations):
        cuda.synchronize()
        start = time.perf_counter()
        shared_bank_stride_test[blocks_per_grid, threads_per_block](d_out, stride)
        cuda.synchronize()
        durations_ms.append((time.perf_counter() - start) * 1000)

    execution_times_ms[stride] = sum(durations_ms) / len(durations_ms)
    print(f"stride = {stride:2d} | temps moyen = {execution_times_ms[stride]:.4f} ms")

print("Q3 : mesures terminées pour", len(strides), "strides.")

stride =  1 | temps moyen = 0.0655 ms
stride =  2 | temps moyen = 0.0841 ms
stride =  4 | temps moyen = 0.0716 ms
stride =  8 | temps moyen = 0.0630 ms
stride = 16 | temps moyen = 0.0592 ms
stride = 32 | temps moyen = 0.0575 ms
Q3 : mesures terminées pour 6 strides.


In [39]:
print("Q4 : évolution du temps en fonction du stride")
print("stride | temps moyen (ms) | conflits théoriques (voies)")
print("-" * 57)
for stride in strides:
    print(f"{stride:>6} | {execution_times_ms[stride]:>16.4f} | {min(stride, 32):>26}")

print("\nConclusion attendue en théorie : plus de conflits bancaires = plus de sérialisation.")
print("Attention : le temps mesuré peut ne pas suivre cet ordre avec le code de l'énoncé.")

Q4 : évolution du temps en fonction du stride
stride | temps moyen (ms) | conflits théoriques (voies)
---------------------------------------------------------
     1 |           0.0655 |                          1
     2 |           0.0841 |                          2
     4 |           0.0716 |                          4
     8 |           0.0630 |                          8
    16 |           0.0592 |                         16
    32 |           0.0575 |                         32

Conclusion attendue en théorie : plus de conflits bancaires = plus de sérialisation.
Attention : le temps mesuré peut ne pas suivre cet ordre avec le code de l'énoncé.


In [40]:
print("Q5 : visualisation des banques pour stride 1 et stride 32")
for stride in [1, 32]:
    indices = [(tid * stride) % 1024 for tid in range(32)]
    banks = [index % 32 for index in indices]
    print(f"\nstride = {stride}")
    print("Indices (32 threads) :", indices)
    print("Banques              :", banks)
    print("Nombre de banques utilisées :", len(set(banks)))

Q5 : visualisation des banques pour stride 1 et stride 32

stride = 1
Indices (32 threads) : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]
Banques              : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]
Nombre de banques utilisées : 32

stride = 32
Indices (32 threads) : [0, 32, 64, 96, 128, 160, 192, 224, 256, 288, 320, 352, 384, 416, 448, 480, 512, 544, 576, 608, 640, 672, 704, 736, 768, 800, 832, 864, 896, 928, 960, 992]
Banques              : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Nombre de banques utilisées : 1


In [41]:
@cuda.jit
def shared_bank_stride_test_validated(d_out32, stride):
    s = cuda.shared.array(1024, dtype=np.float32)
    tid = cuda.threadIdx.x

    # Chaque thread initialise 32 éléments : 32 * 32 = 1024
    for j in range(32):
        index = tid + 32 * j
        s[index] = index * 1.0
    cuda.syncthreads()

    val = np.float32(0.0)
    for i in range(1000):
        # Une autre adresse toutes les itérations,
        # mais le numéro de banque ne change pas pour le thread.
        index = (tid * stride + 32 * i) % 1024
        val += s[index]

    # Rend observable le travail des 32 threads.
    d_out32[tid] = val

# Mesure complémentaire ; la version officielle est au-dessus.
d_out32 = cuda.device_array(32, dtype=np.float32)
print("Bonus : version alternative définie (32 sorties).")
for stride in [1, 2, 4, 8, 16, 32]:
    shared_bank_stride_test_validated[1, 32](d_out32, stride)
    cuda.synchronize()
    durations_ms = []
    for _ in range(30):
        t0 = time.perf_counter()
        shared_bank_stride_test_validated[1, 32](d_out32, stride)
        cuda.synchronize()
        durations_ms.append((time.perf_counter() - t0) * 1000)
    print(f"Bonus stride={stride:2d} : {sum(durations_ms)/len(durations_ms):.4f} ms")

Bonus : version alternative définie (32 sorties).


c:\Users\GiGi\Desktop\TPAML\.venv\Lib\site-packages\numba_cuda\numba\cuda\dispatcher.py:748: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Bonus stride= 1 : 0.0624 ms
Bonus stride= 2 : 0.0659 ms
Bonus stride= 4 : 0.0715 ms
Bonus stride= 8 : 0.0554 ms
Bonus stride=16 : 0.0665 ms
Bonus stride=32 : 0.1081 ms


2D Gaussian Filter in Global Memory

In [42]:
import numpy as np
import matplotlib.pyplot as plt
import time
from numba import cuda

print("NumPy disponible")
print("CUDA disponible :", cuda.is_available())
if not cuda.is_available():
    print("Attention : les questions 3 et 4 nécessitent un GPU NVIDIA et CUDA.")

NumPy disponible
CUDA disponible : True
